# 🔍 Solar Panel Dataset Verification

**เป้าหมาย:** ตอบคำถาม "Aerial datasets เห็น soiling/debris ได้จริงไหม?" ก่อนตัดสินใจ commit กับ strategy

**Datasets ที่จะ verify:**
1. **Jiang PV01** (Zenodo) — UAV 0.1m, panel segmentation only
2. **IA-Cobotics Soiling** — UAV soiling labels
3. **Roboflow Solar Defects** — RGB defects

**วิธีใช้:**
1. เปิดใน Google Colab
2. รัน cell ตามลำดับ
3. ตอบ checklist ตอนท้าย

---

## 📦 Cell 1: Setup

In [ ]:
# Install dependencies (Colab มี packages พื้นฐานอยู่แล้ว แต่เผื่อขาด)
!pip install -q rasterio geopandas

import os
import zipfile
import json
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# สร้าง working directory
WORK_DIR = Path('/content/datasets')
WORK_DIR.mkdir(exist_ok=True)
print(f'Working directory: {WORK_DIR}')

## 📥 Cell 2: Download Jiang PV01 (Zenodo)

Dataset นี้เป็น panel segmentation จาก UAV resolution 0.1m

**หมายเหตุ:** ไฟล์ใหญ่ ~500MB+ — ใช้ Colab จะเร็วกว่า local

In [ ]:
# ดาวน์โหลด PV01 (UAV 0.1m) จาก Zenodo
# ถ้า URL เปลี่ยน ให้เช็คที่ https://zenodo.org/records/5171712

import os
os.chdir(WORK_DIR)

# ลอง wget — ถ้าไม่ได้ลอง curl
!wget -q --show-progress https://zenodo.org/records/5171712/files/PV01.zip -O PV01.zip

# ตรวจสอบไฟล์
!ls -lh PV01.zip

In [ ]:
# Extract
if Path('PV01.zip').exists() and Path('PV01.zip').stat().st_size > 1000:
    !unzip -q -o PV01.zip -d PV01
    print('✅ Extracted')
    !ls PV01 | head -20
else:
    print('❌ Download failed — ลองดาวน์โหลด manual จาก https://zenodo.org/records/5171712')

## 🔎 Cell 3: Inspect Jiang PV01 Structure

In [ ]:
# ดูโครงสร้างโฟลเดอร์
import subprocess
result = subprocess.run(['find', 'PV01', '-maxdepth', '3', '-type', 'd'],
                       capture_output=True, text=True)
print('📁 Folder structure:')
print(result.stdout)

# นับไฟล์แต่ละประเภท
print('\n📊 File counts:')
for ext in ['.bmp', '.png', '.jpg', '.tif', '.tiff', '.shp', '.json']:
    result = subprocess.run(['find', 'PV01', '-iname', f'*{ext}'],
                           capture_output=True, text=True)
    count = len([l for l in result.stdout.split('\n') if l])
    if count > 0:
        print(f'   {ext}: {count} files')

In [ ]:
# หาภาพและ mask ตัวอย่าง
img_extensions = {'.bmp', '.png', '.jpg', '.tif', '.tiff'}
all_files = list(Path('PV01').rglob('*'))
image_files = [f for f in all_files if f.suffix.lower() in img_extensions]

print(f'Found {len(image_files)} image files')
if image_files:
    print('\nตัวอย่าง 10 ไฟล์แรก:')
    for f in image_files[:10]:
        print(f'  {f.relative_to(Path("PV01"))}')

## 🖼️ Cell 4: Visual Inspection — Jiang PV01

**คำถามสำคัญที่ต้องตอบจากการดูภาพ:**
- Resolution 0.1m พอที่จะเห็น soiling/debris บนแผงไหม?
- ถ้าจะ label ปัญหาเอง บนภาพแบบนี้ — เป็นไปได้ไหม?

In [ ]:
def display_image_grid(image_paths, title='Sample images', n=9, figsize=(15, 15)):
    """แสดง grid ของภาพ พร้อมข้อมูล size"""
    n = min(n, len(image_paths))
    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    fig.suptitle(title, fontsize=14, y=1.02)
    axes = np.array(axes).flatten() if rows > 1 else np.atleast_1d(axes).flatten()

    for i, img_path in enumerate(image_paths[:n]):
        try:
            img = Image.open(img_path)
            axes[i].imshow(img)
            axes[i].set_title(f'{img_path.name}\n{img.size[0]}×{img.size[1]}', fontsize=9)
            axes[i].axis('off')
        except Exception as e:
            axes[i].text(0.5, 0.5, f'Error', ha='center')
            axes[i].axis('off')

    for i in range(n, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# แสดงภาพ 9 ใบแรก
display_image_grid(image_files, title='Jiang PV01 — Sample Images', n=9)

In [ ]:
# Stats ของภาพ
sizes = []
for img_path in image_files[:100]:  # sample 100
    try:
        with Image.open(img_path) as img:
            sizes.append(img.size)
    except:
        pass

if sizes:
    widths = [s[0] for s in sizes]
    heights = [s[1] for s in sizes]
    print(f'📐 Image dimensions (sample of {len(sizes)}):')
    print(f'   Width:  min={min(widths)}, max={max(widths)}, avg={np.mean(widths):.0f}')
    print(f'   Height: min={min(heights)}, max={max(heights)}, avg={np.mean(heights):.0f}')
    common = Counter([f'{w}×{h}' for w,h in sizes]).most_common(3)
    print(f'   Common sizes: {common}')

## 🔬 Cell 5: Zoom-in Test — เห็นรายละเอียดบนแผงไหม?

ดูภาพในขนาดเต็ม + crop ส่วนที่มีแผง เพื่อดูว่าเห็นรายละเอียดได้แค่ไหน

In [ ]:
# เลือกภาพ 1 ใบ ดูเต็ม + zoom
if image_files:
    sample_idx = 0  # เปลี่ยนได้
    img_path = image_files[sample_idx]
    img = Image.open(img_path)
    img_arr = np.array(img)

    fig, axes = plt.subplots(1, 2, figsize=(20, 10))

    # ภาพเต็ม
    axes[0].imshow(img_arr)
    axes[0].set_title(f'Full image: {img.size[0]}×{img.size[1]}\n{img_path.name}')
    axes[0].axis('off')

    # Zoom กลางภาพ 25%
    h, w = img_arr.shape[:2]
    crop = img_arr[h//4:3*h//4, w//4:3*w//4]
    axes[1].imshow(crop)
    axes[1].set_title(f'Center crop (50%): {crop.shape[1]}×{crop.shape[0]}')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    print(f'\n💡 ดูภาพแล้วประเมิน:')
    print(f'   - แผงแต่ละใบกินพื้นที่กี่ pixels?')
    print(f'   - ถ้ามี soiling/debris จะมองเห็นได้ไหม?')
    print(f'   - เปลี่ยน sample_idx เพื่อดูภาพอื่น (มีทั้งหมด {len(image_files)} ภาพ)')

## 🦠 Cell 6: Try IA-Cobotics Soiling Dataset

Dataset นี้ตรง scope มากที่สุด — UAV RGB + soiling labels  
**Source:** Said et al. 2025, MDPI Sensors (https://www.mdpi.com/1424-8220/25/3/738)

**หมายเหตุ:** เว็บไซต์อาจต้อง register หรือกรอกฟอร์มก่อนดาวน์โหลด — ลองดูที่:
- https://www.ia-cobotics.com/Soiling-dataset
- ถ้าโหลดไม่ได้ ใช้ option ในเซลล์ถัดไปเป็น alternative

In [ ]:
# ลอง download IA-Cobotics — อาจต้องเปลี่ยน URL ตาม actual link ในเว็บ
# ถ้าโหลดไม่สำเร็จ ให้ดาวน์โหลดด้วยตัวเอง แล้วอัปโหลดเข้า Colab

import os
os.chdir(WORK_DIR)

# Placeholder — เปลี่ยน URL ตาม actual download link
# !wget -q --show-progress <ACTUAL_URL_HERE> -O ia_cobotics.zip

print('⚠️  IA-Cobotics ต้องดาวน์โหลด manual:')
print('    1. ไปที่ https://www.ia-cobotics.com/Soiling-dataset')
print('    2. ดาวน์โหลด zip file')
print('    3. อัปโหลดเข้า Colab โดยลาก-วางลง /content/datasets/')
print('    4. รัน cell ถัดไป')

In [ ]:
# Extract และดูภาพ IA-Cobotics (ถ้ามี)
ia_zip = list(WORK_DIR.glob('*ia*cobotics*.zip')) + list(WORK_DIR.glob('*soiling*.zip'))

if ia_zip:
    zip_path = ia_zip[0]
    print(f'พบไฟล์: {zip_path}')
    !unzip -q -o {zip_path} -d ia_cobotics

    # หาภาพ
    ia_images = [f for f in Path('ia_cobotics').rglob('*')
                 if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}]
    print(f'พบ {len(ia_images)} ภาพ')

    if ia_images:
        display_image_grid(ia_images, title='IA-Cobotics Soiling Dataset', n=9)
else:
    print('ยังไม่มีไฟล์ IA-Cobotics — ดาวน์โหลด manual แล้วรันเซลล์นี้ใหม่')

## 🎯 Cell 7: Roboflow Solar Defects Dataset

**Method:** Roboflow ใช้ API key ในการดาวน์โหลด — ต้องมี account

ขั้นตอน:
1. สมัคร Roboflow ฟรีที่ https://roboflow.com
2. ค้นหา 'solar panel defects' ใน Universe
3. กด Download dataset → COCO format → Show download code
4. Copy code มาใส่ในเซลล์ข้างล่าง

In [ ]:
# Template — แทนที่ด้วย code จาก Roboflow
# !pip install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("WORKSPACE_NAME").project("PROJECT_NAME")
# dataset = project.version(VERSION).download("coco")

print('📌 ดู instruction ในเซลล์ markdown ด้านบน')

## ✅ Cell 8: Verification Checklist

หลังจากดูภาพแล้ว ตอบคำถามนี้สำหรับ **แต่ละ dataset** ที่ verify ได้:

### 🔵 Jiang PV01
- [ ] เห็นแผงแต่ละใบชัดเจน (boundary แยกได้)?
- [ ] แผง 1 ใบ ≥ 200×200 pixels?
- [ ] **ถ้ามี soiling จะเห็นไหม?** (ถ้าไม่มีในภาพ ให้จินตนาการ)
- [ ] **ถ้ามี debris จะเห็นไหม?**
- [ ] License ใช้ได้สำหรับงานวิจัย? (CC-BY 4.0)

### 🟡 IA-Cobotics
- [ ] ดาวน์โหลดได้สำเร็จไหม?
- [ ] เห็น soiling annotations ชัดเจน?
- [ ] Format ของ annotation (bbox/mask/json)?
- [ ] License?

### 🟢 Roboflow Defects
- [ ] หา dataset ที่ตรงโจทย์ใน Universe?
- [ ] ดาวน์โหลดได้?
- [ ] Class ตรงกับที่ต้องการไหม (dust/debris/...)?

### 🎯 ข้อสรุปสุดท้าย
- [ ] Resolution พอจริงไหม?
- [ ] รวม datasets แล้วได้สามคลาส (clean/soiling/debris) เพียงพอไหม?
- [ ] ต้องการ annotate เพิ่มกี่ภาพ?

## 📋 Cell 9: Generate Verification Report

รันเซลล์นี้เพื่อสรุปผลที่ค้นพบ — ใช้ paste ใน notes หรือคุยกับ mentor

In [ ]:
report = '''
🔍 DATASET VERIFICATION REPORT
================================

📦 Jiang PV01
   Total images: ___
   Common size: ___×___
   Panel visible: yes / no
   Soiling visible (if any): yes / no / N/A
   License: CC-BY 4.0

📦 IA-Cobotics
   Total images: ___
   Soiling labels available: yes / no
   Format: bbox / mask / classification
   License: ___

📦 Roboflow Defects
   Total images: ___
   Classes: ___
   Format: COCO
   License: ___

🎯 CONCLUSION:
   Resolution sufficient: yes / no
   Strategy: 
     [ ] Use existing labels + add own annotations
     [ ] Annotate everything from scratch
     [ ] Mix datasets for unified training

   Estimated annotation work: ___ images
   Next step: ___
'''

print(report)